# Clasificacion de urgencia con embeddings multilingues + Logistic Regression.
Mismos datos y mismo split que 02_baseline.py, para comparar de forma justa.

In [1]:
import os
import joblib
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

load_dotenv()

MODELO_EMBEDDINGS = "paraphrase-multilingual-mpnet-base-v2"
VERSION = "v1"

/home/jorge/proyectos/sicpc-thesis/env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 1. Cargar datos (identico al baseline)

In [3]:
os.chdir("..")  # sube un nivel, a la raiz del proyecto
print(os.getcwd())
df_kaggle = pd.read_csv("data/processed/kaggle_es_baseline.csv")
df_kaggle["texto"] = df_kaggle["subject"].fillna("") + " " + df_kaggle["body"].fillna("")
df_kaggle = df_kaggle[["texto", "urgencia"]]
df_kaggle["fuente"] = "kaggle"

df_reales = pd.DataFrame(columns=["texto", "urgencia", "fuente"])
try:
    import mysql.connector

    conn = mysql.connector.connect(
        host="localhost", port=3306,
        user=os.getenv("DB_USER", "sJorge"), password=os.getenv("DB_PASSWORD"),
        database="sicpc",
    )
    df_reales = pd.read_sql(
        "SELECT asunto_anonimizado, cuerpo_anonimizado, urgencia_real "
        "FROM correo WHERE urgencia_real IS NOT NULL",
        conn,
    )
    conn.close()

    if len(df_reales) > 0:
        df_reales["texto"] = (
            df_reales["asunto_anonimizado"].fillna("") + " " +
            df_reales["cuerpo_anonimizado"].fillna("")
        )
        df_reales = df_reales.rename(columns={"urgencia_real": "urgencia"})
        df_reales = df_reales[["texto", "urgencia"]]
        df_reales["fuente"] = "real"
except Exception as e:
    print(f"No se pudieron cargar correos reales: {e}")

df = pd.concat([df_kaggle, df_reales], ignore_index=True)
df = df.dropna(subset=["texto", "urgencia"])
df = df[df["texto"].str.strip() != ""]

print(f"Total: {len(df)} filas")
print(df["fuente"].value_counts())


/home/jorge/proyectos/sicpc-thesis
Total: 962 filas
fuente
kaggle    812
real      150
Name: count, dtype: int64


/tmp/ipykernel_34745/3444991047.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_reales = pd.read_sql(


#### 2. Split (mismo que el baseline)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    df["texto"], df["urgencia"],
    test_size=0.2, random_state=42, stratify=df["urgencia"],
)

#### 3. Embeddings (el modelo NO se reentrena)

In [5]:
print(f"\nCargando modelo de embeddings: {MODELO_EMBEDDINGS}")
encoder = SentenceTransformer(MODELO_EMBEDDINGS)

print("Generando embeddings de entrenamiento...")
X_train_emb = encoder.encode(
    X_train.tolist(), batch_size=32, show_progress_bar=True,
    normalize_embeddings=True,
)
print("Generando embeddings de prueba...")
X_test_emb = encoder.encode(
    X_test.tolist(), batch_size=32, show_progress_bar=True,
    normalize_embeddings=True,
)


Cargando modelo de embeddings: paraphrase-multilingual-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1355.55it/s]


Generando embeddings de entrenamiento...


Batches: 100%|██████████| 25/25 [01:56<00:00,  4.67s/it]


Generando embeddings de prueba...


Batches: 100%|██████████| 7/7 [00:29<00:00,  4.27s/it]


#### 4. Clasificador

In [6]:
modelo = LogisticRegression(max_iter=2000, class_weight="balanced")
modelo.fit(X_train_emb, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'deprecated'


#### 5. Evaluacion

In [7]:
y_pred = modelo.predict(X_test_emb)

print("\n" + "=" * 60)
print("REPORTE DE CLASIFICACION (embeddings)")
print("=" * 60)
print(classification_report(y_test, y_pred))

print("MATRIZ DE CONFUSION")
print(confusion_matrix(y_test, y_pred, labels=["alta", "media", "baja"]))
print("(orden de filas/columnas: alta, media, baja)")

# Metricas por fuente (Kaggle vs correos reales)
fuente_test = df.loc[X_test.index, "fuente"]
for f in ["kaggle", "real"]:
    mask = (fuente_test == f).values
    if mask.sum() > 0:
        print(f"\n--- Solo {f} (n={mask.sum()}) ---")
        print(classification_report(y_test[mask], y_pred[mask]))


REPORTE DE CLASIFICACION (embeddings)
              precision    recall  f1-score   support

        alta       0.78      0.67      0.72        73
        baja       0.50      0.62      0.55        47
       media       0.56      0.55      0.55        73

    accuracy                           0.61       193
   macro avg       0.61      0.61      0.61       193
weighted avg       0.63      0.61      0.62       193

MATRIZ DE CONFUSION
[[49 17  7]
 [11 40 22]
 [ 3 15 29]]
(orden de filas/columnas: alta, media, baja)

--- Solo kaggle (n=159) ---
              precision    recall  f1-score   support

        alta       0.77      0.71      0.74        66
        baja       0.50      0.50      0.50        34
       media       0.53      0.58      0.55        59

    accuracy                           0.62       159
   macro avg       0.60      0.60      0.60       159
weighted avg       0.62      0.62      0.62       159


--- Solo real (n=34) ---
              precision    recall  f1-scor

#### 6. Guardar clasificador + nombre del encoder (van siempre juntos)

In [8]:
os.makedirs("models", exist_ok=True)
joblib.dump(
    {"clasificador": modelo, "encoder": MODELO_EMBEDDINGS},
    f"models/embeddings_{VERSION}.pkl",
)
print(f"\nGuardado en models/embeddings_{VERSION}.pkl")


Guardado en models/embeddings_v1.pkl


In [9]:
res = pd.DataFrame({"texto": X_test.values, "real": y_test.values,
                    "pred": y_pred, "fuente": fuente_test.values})
errores = res[(res["fuente"] == "real") & (res["real"] != res["pred"])]
for _, r in errores.iterrows():
    print(f"REAL={r['real']} PRED={r['pred']} | {r['texto'][:200]}\n")

REAL=media PRED=baja | Tactiq: Meeting Transcription Your Transcript is ready


Meeting Transcription  has been securely saved. 

Open your transcript to generate an AI summary or action items

REAL=media PRED=baja | Compartiste algunos datos de tu Cuenta de Google con distribuidorapop.com.ar Hacer un seguimiento de los datos de tu Cuenta de Google



[EMAIL]

 

<!--[if !mso]><!-->

<!--[if false]><!-->

REAL=alta PRED=media | Factura Nº: 0009-06088278 Estimado [PERSONA], 

 Se adjunta la factura numero 0009-06088278, pago en la fecha 09/28/2026 12:10:31.

REAL=alta PRED=baja | Movistar te acerca tu factura Código de pago electrónico / Link / [DIRECCION]:
[TELEFONO]




 




¡HOLA [PERSONA]!




Movistar te acerca la factura




 









Ven

REAL=media PRED=baja | [PERSONA], Su receta electronica | [DIRECCION]  
			
				
					 Sede Central
					 Riobamba 841
					 Lunes a viernes de 8 a 19hs.
					 www.centrocharles.com.ar
				
			
		
		
				
		
		

REAL=media PRED=baja | ¡Se ha c

#### Verifico si el kernel sigue en memoria con los datos, si da Error ejecutar todo nuevamente

In [10]:
print(len(df), len(y_pred))

962 193


##### Celda A. Muestra cómo se distribuyen las predicciones en el test de correos reales y lista los errores.

In [ ]:
fuente_test = df.loc[X_test.index, "fuente"]
res = pd.DataFrame({
    "texto": X_test.values,
    "real": y_test.values,
    "pred": y_pred,
    "fuente": fuente_test.values,
})
reales = res[res["fuente"] == "real"]

print(f"Test de correos reales (n={len(reales)})\n")
print(pd.DataFrame({
    "verdad": reales["real"].value_counts(),
    "predicho": reales["pred"].value_counts(),
}).fillna(0).astype(int))

print("\nMatriz de confusión (filas = real, columnas = predicho)")
print(pd.crosstab(reales["real"], reales["pred"]))

errores = reales[reales["real"] != reales["pred"]]
print(f"\nErrores: {len(errores)} de {len(reales)}\n")
for _, r in errores.iterrows():
    texto = " ".join(r["texto"].split())[:200]
    print(f"REAL={r['real']} | PRED={r['pred']} | {texto}\n")

##### Celda B. Mide si los correos de urgencia alta traen palabras explícitas de urgencia, comparando Kaggle contra los correos reales sobre todos los datos, no solo el test.

In [ ]:
SENALES = r"urgente|urgencia|hoy|cuanto antes|lo antes posible|a la brevedad|inmediat|vence|vencimiento|plazo|asap|cr[ií]tico"
df["tiene_senal"] = df["texto"].str.contains(SENALES, case=False, regex=True)

print(df.groupby(["fuente", "urgencia"])["tiene_senal"]
        .agg(["mean", "count"]).round(2))